<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Пользовательские-функции" data-toc-modified-id="Пользовательские-функции-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Пользовательские функции</a></span></li><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Подготовка</a></span><ul class="toc-item"><li><span><a href="#Изучим-дисбаланс-классов" data-toc-modified-id="Изучим-дисбаланс-классов-2.1"><span class="toc-item-num">2.1&nbsp;&nbsp;</span>Изучим дисбаланс классов</a></span></li><li><span><a href="#Лемматизация-текста" data-toc-modified-id="Лемматизация-текста-2.2"><span class="toc-item-num">2.2&nbsp;&nbsp;</span>Лемматизация текста</a></span></li><li><span><a href="#Промежуточный-вывод" data-toc-modified-id="Промежуточный-вывод-2.3"><span class="toc-item-num">2.3&nbsp;&nbsp;</span>Промежуточный вывод</a></span></li></ul></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Обучение</a></span><ul class="toc-item"><li><span><a href="#Обучение-базовой-модели" data-toc-modified-id="Обучение-базовой-модели-3.1"><span class="toc-item-num">3.1&nbsp;&nbsp;</span>Обучение базовой модели</a></span><ul class="toc-item"><li><span><a href="#Промежуточный-вывод" data-toc-modified-id="Промежуточный-вывод-3.1.1"><span class="toc-item-num">3.1.1&nbsp;&nbsp;</span>Промежуточный вывод</a></span></li></ul></li><li><span><a href="#Обучение-модели-CatBoost" data-toc-modified-id="Обучение-модели-CatBoost-3.2"><span class="toc-item-num">3.2&nbsp;&nbsp;</span>Обучение модели CatBoost</a></span><ul class="toc-item"><li><span><a href="#Промежуточный-выввод" data-toc-modified-id="Промежуточный-выввод-3.2.1"><span class="toc-item-num">3.2.1&nbsp;&nbsp;</span>Промежуточный выввод</a></span></li></ul></li></ul></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Выводы</a></span></li><li><span><a href="#Чек-лист-проверки" data-toc-modified-id="Чек-лист-проверки-5"><span class="toc-item-num">5&nbsp;&nbsp;</span>Чек-лист проверки</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

In [ ]:
!pip install transformers -q

In [ ]:
!pip install optuna -q

In [ ]:
!pip install optuna_integration -q

In [ ]:
pip install imbalanced-learn -q

In [ ]:
!pip install --upgrade scikit-learn -q

In [ ]:
!pip install spacy -q

In [ ]:
!pip install en-core-web-sm -q

In [ ]:
!pip install --upgrade wordcloud -q

In [ ]:
!pip install --upgrade pillow -q

In [ ]:
!pip install tqdm -q

In [1]:
import time

import pandas as pd
import numpy as np

from tqdm import tqdm

from torch.utils.data import TensorDataset, DataLoader
import transformers

import spacy

import re

import nltk
from nltk.corpus import stopwords as nltk_stopwords
from sklearn.feature_extraction.text import TfidfVectorizer 

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud, STOPWORDS

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from numpy.random import RandomState
from numpy import mean
from numpy import std

import optuna
from optuna_integration import CatBoostPruningCallback

# загружаем класс pipeline
from imblearn.pipeline import Pipeline

# импортируем класс RandomizedSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit

# загружаем нужные модели
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from catboost import CatBoostClassifier, Pool, cv

ModuleNotFoundError: No module named 'spacy'

In [ ]:
RANDOM_STATE= 57
state = RandomState(57)
TEST_SIZE= 0.1
plt.rcParams["figure.figsize"] = (12,6)

## Пользовательские функции

In [ ]:
def df_info(df):
	if isinstance(df, dict):
		for key, value in df.items():
			print(key)
			value.info()
			display(value.head(5))
			print('----------------------------------------------------')
			print('\n')
	else:
		df.info()
		display(df.head(5))
		print('----------------------------------------------------')
    

## Подготовка

In [ ]:
try:
    comments = pd.read_csv('toxic_comments.csv', 
                           index_col=[0],
                        #    encoding='utf-8' # cp1251
                           ).sort_index()
except:
    comments = pd.read_csv('/datasets/toxic_comments.csv', 
                           index_col=[0],
                        #    encoding='utf-8'
                           ).sort_index()
print(comments.shape)

### Изучим дисбаланс классов

In [ ]:
sns.countplot(comments['toxic'])
plt.title('Дисбаланс классов')
plt.show()

Согласно графику в данных присутствует значительный дисбаланс классов

### Лемматизация текста

In [ ]:
nlp = spacy.load('en_core_web_sm')
def lemmatize(text):
    doc = nlp(text)
    lemmas = [token.lemma_ for token in doc]
    lemm_text = " ".join(lemmas)
    return lemm_text

def clear_text(text):
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = ' '.join(text.split())
    text = re.sub(r'don t', 'dont', text, flags=re.IGNORECASE)
    return text

In [ ]:
df_info(comments)

In [ ]:
train, test = train_test_split(comments, random_state=RANDOM_STATE, test_size=TEST_SIZE)

In [ ]:
tqdm.pandas()
X_train = train['text'].progress_apply(lambda x: lemmatize(clear_text(x)))
X_test = test['text'].progress_apply(lambda x: lemmatize(clear_text(x)))
y_train = train['toxic']
y_test = test['toxic']

nltk.download('stopwords')
stopwords = list(nltk_stopwords.words('english'))

In [ ]:
all_text = ' '.join(X_train)

In [ ]:
wordCloud = WordCloud(random_state=RANDOM_STATE,
                      max_words=100).generate(all_text)
plt.figure(figsize=(12,12))
plt.imshow(wordCloud)

### Промежуточный вывод

По итогу подготовки данных был выявлен значительный дисбаланс классов, который необходимо будет учитывать при обучении моделей.

Также были подготовлены тренировочные и тестовые выборки с использованием лемматизации.

## Обучение

### Обучение базовой модели

In [ ]:
pipe_final = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words=stopwords)),
    ('models', LogisticRegression(class_weight='balanced',
                                 random_state=RANDOM_STATE))
])

param_grid = [
    {
        'models': [LogisticRegression(class_weight='balanced',
                                     random_state=RANDOM_STATE)],
        'models__C': [0.2,0.4,0.6,0.8,1,1.2,1.4]
    },
        
	{
        'models': [DecisionTreeClassifier(class_weight='balanced',
                                         random_state=RANDOM_STATE)],
        'models__max_depth': [6,10,14,20,30],
    }
]

In [ ]:
randomized_search = RandomizedSearchCV(
    pipe_final, 
    param_grid, 
    cv=5,
    scoring='f1',
    refit='f1',
    n_jobs=-1,
    verbose=1,
    random_state=RANDOM_STATE,
    n_iter=20
)

In [ ]:
randomized_search.fit(X_train, y_train)
base_model = randomized_search.best_estimator_
name_base = type(base_model.named_steps['models']).__name__
base_score = randomized_search.best_score_

print('Лучшая модель и её параметры:\n\n', randomized_search.best_estimator_)
print ('Метрика F1 лучшей модели на тренировочной выборке:', randomized_search.best_score_)

#### Промежуточный вывод

Метрика F1 на кроссвалидации Логистической регрессии показала среднее значение 0.7515.

### Обучение модели CatBoost

In [ ]:
cv_dataset = Pool(data=X_train,
                  label=y_train,
                  text_features=[0]
                  )

In [ ]:
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 600, step=100),
        "depth": trial.suggest_int("depth", 8, 15),
        "learning_rate": trial.suggest_float("learning_rate", 0.1, 0.6, log=True),
        "loss_function": "Logloss",
        "eval_metric": "F1",
        "verbose": False,
        "random_seed": RANDOM_STATE,
        "early_stopping_rounds": 50,
		"auto_class_weights": 'Balanced'
    }
    
    # Callback для прунинга
    pruning_callback = CatBoostPruningCallback(trial, "f1")  # Укажите метрику для отслеживания
    
    # Выполнение кросс-валидации
    cv_results = cv(
        pool=cv_dataset,
        params=params,
        fold_count=5,
        partition_random_seed=RANDOM_STATE,
        shuffle=True,
        plot=False,
        stratified=True,     # Использовать стратифицированные фолды
        verbose=False
    )
    
	# Проверка, не было ли испытание прервано
    pruning_callback.check_pruned()
    
    # Возвращаем среднее значение F1-меры
    return cv_results["test-F1-mean"].mean()

In [ ]:
%%time
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study.optimize(objective, n_trials=20, show_progress_bar=True)
catboost_best_value = study.best_value

print(f"Лучшие параметры: {study.best_params}")
print(f"Лучшее значение F1: {catboost_best_value:.4f}")

In [ ]:
fixed_params = {
        "loss_function": "Logloss",
        "eval_metric": "F1",
        "verbose": False,
        "random_seed": RANDOM_STATE,
        "early_stopping_rounds": 50,
		"auto_class_weights": 'Balanced'}
try:
    catboost_best_params = {**fixed_params, **study.best_params}
except:
	catboost_best_params = {'iterations': 500, #260
							'depth': 12, #10
							'learning_rate': 0.18, #0.237
							**fixed_params}

catboost_model = CatBoostClassifier(**catboost_best_params)
print(catboost_best_params)

In [ ]:
scores = cv(cv_dataset,
            catboost_best_params,
            fold_count=3,
            shuffle=True,
            plot=True)

f1_mean = scores['test-F1-mean'].iloc[-1]
f1_std  = scores['test-F1-std'].iloc[-1]

catboost_score = f1_mean

print(f'F1 CV: {f1_mean:.4f} ± {f1_std:.4f}')

In [ ]:
%%time
catboost_model.fit(X_train, y_train, text_features=[0], verbose=50, early_stopping_rounds=catboost_best_params['early_stopping_rounds'])

#### Промежуточный выввод

Модель Catboost показала результат F1 на кроссвалидации больше 0.9, что больше целевого значения 0.75. Далее будет тестировать данную модель с заданными гиперпараметрами

In [ ]:
X_test = pd.DataFrame(X_test, columns=['text'])

In [ ]:
X_test.head()

In [ ]:
predicts = catboost_model.predict(X_test)
print(f'{f1_score(y_test,predicts):.4f}')

Результат на тестовой выборке показал метрику F1 0.7575, что выше целевого значения 0.75

## Выводы

По результатам работы была обучена модель CatBoost с метрикой F1 - 0.7575, что выше целевого значения. Модель была обучена на данных приведенных к TF-IDF.